In [ ]:
import os, json, time, math, platform, sys, uuid
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

print({"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
       "tensorflow": tf.__version__, "scikit_learn": sklearn.__version__})


In [ ]:
DATA_ROOT = Path("/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate (Copy)")
OUTPUT_ROOT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", str(Path.cwd() / "DL_JointCORAL_Runs")))
# RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]
# PROJECT_DIR = OUTPUT_ROOT / RUN_ID
# PROJECT_DIR.mkdir(parents=True, exist_ok=False)
CARD_SPLIT_MANIFEST = None
SOURCE_VALIDATION_CARDS = [1, 2, 3, 4, 5]

CLASS_ORDER = [f"B{i}" for i in range(12)]
SEED = 123                                                     
LEARNING_RATE = 0.001                                                                       
LAMBDA_CORAL = 0.1                                                                          
BATCH_SIZE = 32                                           
MAX_EPOCHS_SOURCE = 30                                     
MAX_EPOCHS_ADAPT = 30                                      
EARLY_STOPPING_PATIENCE = 5                                
POOL_SIZE = 4                                                      
POOL_STRIDE = 4                                                     
CONV_PADDING = "same"                                               
CONV_STRIDE = 1                                                     
EMBEDDING_DROPOUT = 0.30                                             
CLASSIFIER_DROPOUT = 0.30                                            

RUN_FULL_EXPERIMENT = False                                                              
ALLOW_KNOWN_OVERLAP_EXPLORATORY = False                                                                 

ADAPTATION_PATHS = {
    "source": {"base": "Source", "cards": list(range(1, 6)), "reader": "R1"},
    "CC": {"base": "CC", "cards": list(range(6, 11)), "reader": "R1"},
    "CR": {"base": "CR_/R2", "cards": list(range(1, 6)), "reader": "R2"},
    "CCR": {"base": "CCR/R2", "cards": list(range(11, 16)), "reader": "R2"},
}
EVALUATION_PATHS = {
    "CC": {"base": "CC", "cards": list(range(16, 26)), "reader": "R1"},
    "CR": {"base": "CR_/R3", "cards": list(range(1, 6)), "reader": "R3"},
    "CCR": {"base": "CCR/R3", "cards": list(range(26, 36)), "reader": "R3"},
}

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root does not exist: {DATA_ROOT}")
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Notebook project directory does not exist: {PROJECT_DIR}")


In [ ]:
def expected_files(spec):
    return [DATA_ROOT / spec["base"] / f"card{card}" / f"button{b}.npy"
            for card in spec["cards"] for b in range(12)]

all_specs = {**{k:v for k,v in ADAPTATION_PATHS.items()}, **{f"eval_{k}":v for k,v in EVALUATION_PATHS.items()}}
inventory_rows=[]
for domain, spec in all_specs.items():
    files=expected_files(spec)
    missing=[str(p) for p in files if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"{domain} missing {len(missing)} expected button files; first: {missing[:3]}")
    for p in files:
        a=np.load(p, mmap_mode="r", allow_pickle=False)
        inventory_rows.append({"domain":domain,"relative_path":str(p.relative_to(DATA_ROOT)),"format":p.suffix,
                               "shape":str(a.shape),"dtype":str(a.dtype),"rows":int(a.shape[0]),
                               "signal_length":int(a.shape[1]) if a.ndim==2 else None,
                               "is_complex":bool(np.iscomplexobj(a)),"bytes":p.stat().st_size})
        if a.ndim!=2 or a.shape[1]!=3624 or not np.iscomplexobj(a):
            raise ValueError(f"Unexpected stored data contract in {p}: {a.shape} {a.dtype}")
INVENTORY=pd.DataFrame(inventory_rows)
display(INVENTORY.groupby("domain").agg(files=("relative_path","count"),rows=("rows","sum"),
       signal_lengths=("signal_length",lambda x: sorted(set(x))),dtypes=("dtype",lambda x: sorted(set(x))),
       complex_files=("is_complex","sum")))


In [ ]:
if CARD_SPLIT_MANIFEST is None:
    if sorted(SOURCE_VALIDATION_CARDS) != [1, 2, 3, 4, 5]:
        raise ValueError("SOURCE_VALIDATION_CARDS must contain each source card exactly once")
    CARD_SPLITS = pd.DataFrame([
        {"run_id": f"run{i+1}", "card": card, "reader": "R1",
         "partition": "validation" if card == validation_card else "train"}
        for i, validation_card in enumerate(SOURCE_VALIDATION_CARDS)
        for card in range(1, 6)
    ])
    SPLIT_ORIGIN = "new_leave_one_source_card_out"
else:
    CARD_SPLIT_MANIFEST = Path(CARD_SPLIT_MANIFEST)
    CARD_SPLITS = pd.read_csv(CARD_SPLIT_MANIFEST)
    SPLIT_ORIGIN = str(CARD_SPLIT_MANIFEST)
required = {"run_id", "card", "reader", "partition"}
if not required.issubset(CARD_SPLITS.columns):
    raise ValueError(f"Split columns required: {sorted(required)}")
CARD_SPLITS["card"] = pd.to_numeric(CARD_SPLITS["card"], errors="raise").astype(int)
CARD_SPLITS["reader"] = CARD_SPLITS["reader"].astype(str).str.strip().str.upper()
CARD_SPLITS["partition"] = CARD_SPLITS["partition"].astype(str).str.strip().str.lower()
FOLD_IDS = sorted(CARD_SPLITS["run_id"].unique())
if len(FOLD_IDS) != 5:
    raise ValueError("Exactly five source-validation folds are required")
FOLD_SOURCE_CARDS = {}
for fold_id in FOLD_IDS:
    q = CARD_SPLITS[(CARD_SPLITS.run_id == fold_id) & (CARD_SPLITS.reader == "R1") & CARD_SPLITS.card.between(1, 5)]
    if len(q) != 5 or q.card.duplicated().any() or set(q.card) != set(range(1,6)):
        raise ValueError(f"Invalid source-card membership: {fold_id}")
    if not q.partition.isin(["train", "validation"]).all() or (q.partition == "validation").sum() != 1:
        raise ValueError(f"One validation card and four training cards required: {fold_id}")
    val_card = int(q.loc[q.partition == "validation", "card"].iloc[0])
    FOLD_SOURCE_CARDS[fold_id] = {"validation_card": val_card, "train_cards": sorted(q.loc[q.partition == "train", "card"].tolist())}
display(CARD_SPLITS)


In [ ]:
def magnitude_rows(complex_rows):
    a=np.asarray(complex_rows)
    if not np.iscomplexobj(a) or not np.isfinite(a).all():
        raise TypeError("Expected stored complex I+jQ responses")
    return np.abs(a).astype(np.float32)

def fit_source_normalizer(source_training_magnitude):
    x=np.asarray(source_training_magnitude,dtype=np.float32)
    if x.ndim!=2 or len(x)==0: raise ValueError("error")
    mu=float(np.mean(x,dtype=np.float64))
    sigma=float(np.std(x,dtype=np.float64))
    if not np.isfinite(mu) or not np.isfinite(sigma) or sigma<=0: raise ValueError("Invalid source-only normalization statistics")
    return {"representation":"abs(I+1j*Q)","normalization":"one global scalar z-score",
            "mean":mu,"std":sigma,"fit_role":"source training responses only",
            "per_response_centering":False,"detrending":False}

def apply_source_normalizer(magnitude, stats):
    x=np.asarray(magnitude,dtype=np.float32)
    if not np.isfinite(x).all() or not np.isfinite(stats["std"]) or stats["std"] <= 0:
        raise ValueError("Invalid magnitude data or normalization standard deviation")
    return ((x-np.float32(stats["mean"]))/np.float32(stats["std"]))[...,None].astype(np.float32)


In [ ]:
def load_domain_magnitudes(domain, cards, evaluation=False):
    spec=(EVALUATION_PATHS if evaluation else ADAPTATION_PATHS)[domain]
    chunks=[]; labels=[]; sample_ids=[]; card_folder=[]; readers=[]; row_order=[]; capture_session=[]
    for card in cards:
        base=DATA_ROOT/spec["base"]/f"card{card}"
        for b in range(12):
            p=base/f"button{b}.npy"
            arr=np.load(p,allow_pickle=False)
            if arr.shape!=(200,3624) or not np.iscomplexobj(arr) or not np.isfinite(arr).all(): raise ValueError(f"Bad array {p}: {arr.shape}")
            chunks.append(magnitude_rows(arr))
            labels.extend([b]*len(arr))
            sample_ids.extend([f"{p.relative_to(DATA_ROOT)}#row={r:04d}" for r in range(len(arr))])
            card_folder.extend([f"card{card}"]*len(arr)); readers.extend([spec["reader"]]*len(arr))
            row_order.extend(list(range(len(arr)))); capture_session.extend([None]*len(arr))
    return {"magnitude":np.concatenate(chunks,axis=0),"y":np.asarray(labels,dtype=np.int64),
            "sample_id":np.asarray(sample_ids,dtype=str),"card_folder":np.asarray(card_folder,dtype=str),
            "reader":np.asarray(readers,dtype=str),"row_order":np.asarray(row_order,dtype=np.int64),
            "capture_session":np.asarray(["" if v is None else str(v) for v in capture_session],dtype=str)}

SAMPLE_FILES={k:expected_files(v)[0] for k,v in all_specs.items()}
for role,p in SAMPLE_FILES.items():
    a=np.load(p,mmap_mode="r",allow_pickle=False)
    print(role,str(p.relative_to(DATA_ROOT)),a.shape,a.dtype,"magnitude sample mean",float(np.abs(a[:2]).mean()))


In [ ]:
def build_joint_models(signal_length=3624, embedding_dropout=EMBEDDING_DROPOUT,
                       classifier_dropout=CLASSIFIER_DROPOUT, pool_size=POOL_SIZE,
                       pool_stride=POOL_STRIDE, conv_padding=CONV_PADDING):
    signal=tf.keras.Input(shape=(signal_length,1),name="magnitude_response")
    x=tf.keras.layers.Conv1D(32,7,strides=CONV_STRIDE,padding=conv_padding,name="conv1_32_k7")(signal)
    x=tf.keras.layers.BatchNormalization(name="conv1_bn")(x)
    x=tf.keras.layers.ReLU(name="conv1_relu")(x)
    x=tf.keras.layers.MaxPooling1D(pool_size=pool_size,strides=pool_stride,name="conv1_pool")(x)
    x=tf.keras.layers.Conv1D(64,7,strides=CONV_STRIDE,padding=conv_padding,name="conv2_64_k7")(x)
    x=tf.keras.layers.BatchNormalization(name="conv2_bn")(x)
    x=tf.keras.layers.ReLU(name="conv2_relu")(x)
    x=tf.keras.layers.MaxPooling1D(pool_size=pool_size,strides=pool_stride,name="conv2_pool")(x)
    x=tf.keras.layers.LSTM(64,return_sequences=True,name="lstm64")(x)
    x=tf.keras.layers.GlobalAveragePooling1D(name="global_average_pool")(x)
    x=tf.keras.layers.Dense(64,activation="relu",name="embedding_dense64")(x)
    z=tf.keras.layers.Dropout(embedding_dropout,name="embedding_dropout")(x)
    encoder=tf.keras.Model(signal,z,name="shared_encoder")

    emb_in=tf.keras.Input(shape=(64,),name="encoder_embedding")
    h=tf.keras.layers.Dense(64,name="classifier_dense64")(emb_in)
    h=tf.keras.layers.ReLU(name="classifier_relu")(h)
    h=tf.keras.layers.BatchNormalization(name="classifier_bn")(h)
    h=tf.keras.layers.Dropout(classifier_dropout,name="classifier_dropout")(h)
    logits=tf.keras.layers.Dense(12,activation=None,name="button_logits")(h)
    classifier=tf.keras.Model(emb_in,logits,name="button_classifier")

    model_in=tf.keras.Input(shape=(signal_length,1),name="model_response")
    model_out=classifier(encoder(model_in))
    full=tf.keras.Model(model_in,model_out,name="joint_deepcoral_pinpad")
    if full.output_shape[-1]!=12 or encoder.output_shape[-1]!=64:
        raise AssertionError("Model output/embedding contract failed")
    return full,encoder,classifier

MODEL_CONFIG={"conv_filters":[32,64],"kernel_sizes":[7,7],"pool_size":POOL_SIZE,"pool_stride":POOL_STRIDE,
 "conv_padding":CONV_PADDING,"conv_stride":CONV_STRIDE,"lstm_units":64,"return_sequences":True,
 "global_pooling":"average","embedding_units":64,"classifier_hidden_units":64,
 "embedding_dropout":EMBEDDING_DROPOUT,"classifier_dropout":CLASSIFIER_DROPOUT,
 "class_order":CLASS_ORDER,"output":"12 logits"}


In [ ]:
def make_optimizer(learning_rate=LEARNING_RATE):
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)

def sparse_ce_from_logits(y_true, logits):
    return tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y_true,logits,from_logits=True))

def source_update(model, optimizer, xb, yb):
    with tf.GradientTape() as tape:
        logits=model(xb,training=True)
        loss=sparse_ce_from_logits(yb,logits)
    grads=tape.gradient(loss,model.trainable_variables)
    if any(g is None for g in grads): raise RuntimeError("A trainable source variable received no classification gradient")
    optimizer.apply_gradients(zip(grads,model.trainable_variables))
    return float(loss.numpy())

def iter_minibatches(n,batch_size,rng,shuffle=True):
    order=rng.permutation(n) if shuffle else np.arange(n)
    for start in range(0,n-batch_size+1,batch_size):
        yield order[start:start+batch_size]

def train_source_model(x_train,y_train,x_val,y_val,output_dir,config=None):
    config=config or {}
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(int(config.get("seed",SEED)))
    model,encoder,classifier=build_joint_models(signal_length=x_train.shape[1])
    optimizer=make_optimizer(config.get("learning_rate",LEARNING_RATE))
    batch_size=int(config.get("batch_size",BATCH_SIZE)); max_epochs=int(config.get("max_epochs",MAX_EPOCHS_SOURCE))
    patience=int(config.get("patience",EARLY_STOPPING_PATIENCE)); history=[]; best_weights=None; best_key=(-float("inf"),-float("inf")); wait=0
    for epoch in range(max_epochs):
        losses=[]; rng=np.random.default_rng(int(config.get("seed",SEED))+epoch)
        for ix in iter_minibatches(len(y_train),batch_size,rng):
            losses.append(source_update(model,optimizer,tf.convert_to_tensor(x_train[ix]),tf.convert_to_tensor(y_train[ix])))
        val_logits=model(x_val,training=False).numpy(); val_pred=val_logits.argmax(axis=1)
        val_acc=float(np.mean(val_pred==y_val)); val_loss=float(sparse_ce_from_logits(tf.convert_to_tensor(y_val),tf.convert_to_tensor(val_logits)).numpy())
        history.append({"epoch":epoch+1,"train_ce":float(np.mean(losses)),"val_ce":val_loss,"val_accuracy":val_acc})
        key=(val_acc,-val_loss)
        if key>best_key:
            best_key=key;best_weights=model.get_weights();best_epoch=epoch+1;wait=0
        else:
            wait+=1
            if wait>=patience: break
    if best_weights is None: raise RuntimeError("No validation-selected source checkpoint")
    model.set_weights(best_weights)
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=False)
    model.save(out/"source_model.keras")
    (out/"source_training_history.json").write_text(json.dumps(history,indent=2)+"\n")
    (out/"source_selection.json").write_text(json.dumps({"partition":"source validation card","criterion":"highest validation accuracy, then lowest validation CE","best_epoch":best_epoch,"best_val_accuracy":best_key[0]},indent=2)+"\n")
    return model,history


In [ ]:
def sample_covariance(features):
    features=tf.convert_to_tensor(features)
    n=tf.shape(features)[0]
    tf.debugging.assert_greater(n,1,message="Sample covariance requires at least two rows")
    centered=features-tf.reduce_mean(features,axis=0,keepdims=True)
    denom=tf.cast(n-1,features.dtype)
    return tf.matmul(centered,centered,transpose_a=True)/denom

def deep_coral_loss(source_features,target_features):
    d=tf.cast(tf.shape(source_features)[1],source_features.dtype)
    delta=sample_covariance(source_features)-sample_covariance(target_features)
    return tf.reduce_sum(tf.square(delta))/(4.0*d*d)

def joint_objective(source_logits,source_labels,z_source,z_cc,z_cr,z_ccr,lambda_coral=LAMBDA_CORAL):
    ce=sparse_ce_from_logits(source_labels,source_logits)
    target_losses=[deep_coral_loss(z_source,z_t) for z_t in (z_cc,z_cr,z_ccr)]
    coral=tf.add_n(target_losses)/3.0
    return ce+tf.cast(lambda_coral,ce.dtype)*coral,ce,coral,target_losses

def joint_update(model,optimizer,x_source,y_source,x_cc,x_cr,x_ccr,lambda_coral=LAMBDA_CORAL):
    batches=[tf.convert_to_tensor(x) for x in (x_source,x_cc,x_cr,x_ccr)]
    sizes=[tf.shape(x)[0] for x in batches]
    for n in sizes[1:]: tf.debugging.assert_equal(sizes[0],n,message="Equal batch size required across source/CC/CR/CCR")
    if len(batches)!=4: raise ValueError("Need source plus CC, CR, CCR")
    with tf.GradientTape() as tape:
        z_s=model.get_layer("shared_encoder")(batches[0],training=True)
        z_cc=model.get_layer("shared_encoder")(batches[1],training=True)
        z_cr=model.get_layer("shared_encoder")(batches[2],training=True)
        z_ccr=model.get_layer("shared_encoder")(batches[3],training=True)
        logits=model.get_layer("button_classifier")(z_s,training=True)
        total,ce,coral,losses=joint_objective(logits,tf.convert_to_tensor(y_source),z_s,z_cc,z_cr,z_ccr,lambda_coral)
    grads=tape.gradient(total,model.trainable_variables)
    if any(g is None for g in grads): raise RuntimeError("Joint objective left a trainable variable without a gradient")
    optimizer.apply_gradients(zip(grads,model.trainable_variables))
    return {"total":float(total.numpy()),"source_ce":float(ce.numpy()),"coral_mean":float(coral.numpy()),
            "coral_CC":float(losses[0].numpy()),"coral_CR":float(losses[1].numpy()),"coral_CCR":float(losses[2].numpy())}


In [ ]:
def adapt_joint_multi_target(source_checkpoint,x_source,y_source,target_train_x,x_val,y_val,
                             output_dir,config=None):
    config=config or {}
    tf.keras.utils.set_random_seed(int(config.get("seed", SEED)))
    if set(target_train_x)!={"CC","CR","CCR"}: raise ValueError("Exactly three targets CC/CR/CCR are required")

    model=tf.keras.models.load_model(source_checkpoint,compile=False)
    batch=int(config.get("batch_size",BATCH_SIZE)); max_epochs=int(config.get("max_epochs",MAX_EPOCHS_ADAPT))
    patience=int(config.get("patience",EARLY_STOPPING_PATIENCE)); lam=float(config.get("lambda_coral",LAMBDA_CORAL))
    arrays={d:np.asarray(target_train_x[d],dtype=np.float32) for d in ("CC","CR","CCR")}
    if min([len(y_source)]+[len(x) for x in arrays.values()])<batch: raise ValueError("At least one equal-sized mini-batch per domain is required")
    optimizer=make_optimizer(config.get("learning_rate",LEARNING_RATE)); history=[]; best_weights=None; best_key=(-float("inf"),-float("inf")); wait=0
    for epoch in range(max_epochs):
        rng=np.random.default_rng(int(config.get("seed",SEED))+10000+epoch)
        src_order=rng.permutation(len(y_source)); tgt_order={d:rng.permutation(len(arrays[d])) for d in arrays}
        steps=min([len(src_order)//batch]+[len(tgt_order[d])//batch for d in arrays])
        records=[]
        for step in range(steps):
            si=src_order[step*batch:(step+1)*batch]
            ti={d:tgt_order[d][step*batch:(step+1)*batch] for d in arrays}
            records.append(joint_update(model,optimizer,x_source[si],y_source[si],arrays["CC"][ti["CC"]],
                                        arrays["CR"][ti["CR"]],arrays["CCR"][ti["CCR"]],lam))
        if not records: raise RuntimeError("No complete joint mini-batch step")
        val_logits=model(x_val,training=False).numpy(); val_acc=float(np.mean(val_logits.argmax(1)==y_val))
        val_ce=float(sparse_ce_from_logits(tf.convert_to_tensor(y_val),tf.convert_to_tensor(val_logits)).numpy())
        row={"epoch":epoch+1,"total_loss":float(np.mean([r["total"] for r in records])),
             "source_ce":float(np.mean([r["source_ce"] for r in records])),"coral_mean":float(np.mean([r["coral_mean"] for r in records])),
             "coral_CC":float(np.mean([r["coral_CC"] for r in records])),"coral_CR":float(np.mean([r["coral_CR"] for r in records])),
             "coral_CCR":float(np.mean([r["coral_CCR"] for r in records])),"source_validation_accuracy":val_acc,"source_validation_ce":val_ce}
        history.append(row); key=(val_acc,-val_ce)
        if key>best_key: best_key=key;best_weights=model.get_weights();best_epoch=epoch+1;wait=0
        else:
            wait+=1
            if wait>=patience: break
    if best_weights is None: raise RuntimeError("Adaptation has no source-validation-selected checkpoint")
    model.set_weights(best_weights)
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=False)
    model.save(out/"adapted_model.keras")
    (out/"adaptation_history.json").write_text(json.dumps(history,indent=2)+"\n")
    (out/"adaptation_selection.json").write_text(json.dumps({"validation":"source validation only","criterion":"highest source validation accuracy, then lowest source validation CE","best_epoch":best_epoch,"best_validation_accuracy":best_key[0]},indent=2)+"\n")
    return model,history


In [ ]:
def prepare_fold(fold_id):
    source_split=FOLD_SOURCE_CARDS[fold_id]
    source_train=load_domain_magnitudes("source",source_split["train_cards"],evaluation=False)
    source_val=load_domain_magnitudes("source",[source_split["validation_card"]],evaluation=False)
    norm=fit_source_normalizer(source_train["magnitude"])
    x_train=apply_source_normalizer(source_train["magnitude"],norm); x_val=apply_source_normalizer(source_val["magnitude"],norm)
    targets={}; target_meta={}
    for d in ("CC","CR","CCR"):
        raw=load_domain_magnitudes(d,ADAPTATION_PATHS[d]["cards"],evaluation=False)
        targets[d]=apply_source_normalizer(raw["magnitude"],norm)

        target_meta[d]={k:v for k,v in raw.items() if k!="magnitude"}
    return {"x_train":x_train,"y_train":source_train["y"],"train_meta":{k:v for k,v in source_train.items() if k!="magnitude"},
            "x_val":x_val,"y_val":source_val["y"],"val_meta":{k:v for k,v in source_val.items() if k!="magnitude"},
            "target_x":targets,"target_audit_metadata":target_meta,"normalization":norm,
            "source_train_cards":source_split["train_cards"],"source_validation_card":source_split["validation_card"]}

KNOWN_OVERLAP_AUDIT={"source_run":"run_20261030T165828Z/data_mapping_and_overlap.json",
 "adaptation_unique_payloads":2600,"evaluation_unique_payloads":2600,"shared_unique_payloads":2600,
 "shared_fraction":1.0,"identity_note":"payload equality alone does not prove acquisition identity"}

def confirm_full_run_allowed():
    if KNOWN_OVERLAP_AUDIT["shared_fraction"]>0 and not ALLOW_KNOWN_OVERLAP_EXPLORATORY:
        raise RuntimeError("Historical adaptation/evaluation payload overlap recorded. Review audit before enabling ALLOW_KNOWN_OVERLAP_EXPLORATORY.")
    return "exploratory_due_to_known_exact_payload_overlap" if KNOWN_OVERLAP_AUDIT["shared_fraction"]>0 else "overlap_not_detected"


In [ ]:
def predict_logits_probabilities_embeddings(model,x,batch_size=128):
    logits_out=[]; probs_out=[]; emb_out=[]
    enc=model.get_layer("shared_encoder"); clf=model.get_layer("button_classifier")
    for start in range(0,len(x),batch_size):
        xb=tf.convert_to_tensor(x[start:start+batch_size])
        z=enc(xb,training=False).numpy()
        logits=clf(z,training=False).numpy()
        prob=tf.nn.softmax(logits,axis=-1).numpy()                                        
        logits_out.append(logits);probs_out.append(prob);emb_out.append(z)
    return np.concatenate(logits_out),np.concatenate(probs_out),np.concatenate(emb_out)

def evaluate_model_on_collection(model,domain,cards,norm,output_dir,model_name):
    raw=load_domain_magnitudes(domain,cards,evaluation=True)
    x=apply_source_normalizer(raw["magnitude"],norm)
    logits,probs,emb=predict_logits_probabilities_embeddings(model,x)
    pred=logits.argmax(axis=1); y=raw["y"]
    cm=confusion_matrix(y,pred,labels=np.arange(12))
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(out/f"{model_name}_{domain}_predictions.npz",sample_ids=raw["sample_id"],true_labels=y,
                        logits=logits,probabilities=probs,predictions=pred,embeddings=emb,
                        class_order=np.asarray(CLASS_ORDER),card_folder=raw["card_folder"],reader=raw["reader"],row_order=raw["row_order"])
    np.savez_compressed(out/f"{model_name}_{domain}_confusion.npz",counts=cm,labels=np.asarray(CLASS_ORDER))
    return {"model":model_name,"domain":domain,"n":int(len(y)),"accuracy":float(accuracy_score(y,pred)),"confusion":cm,
            "sample_ids":raw["sample_id"],"y":y,"pred":pred,"probabilities":probs,"embeddings":emb}

def run_five_card_split_experiment():
    overlap_status=confirm_full_run_allowed()
    run_id=datetime.now(timezone.utc).strftime("full_run_%Y%m%dT%H%M%SZ_")+uuid.uuid4().hex[:6]
    root=PROJECT_DIR/run_id
    if root.exists(): raise FileExistsError(root)
    root.mkdir(parents=True,exist_ok=False)
    CARD_SPLITS.to_csv(root/"source_card_splits.csv", index=False)
    records=[]; histories={}; norms={}
    for i,fold_id in enumerate(FOLD_IDS):
        fold=prepare_fold(fold_id); fold_dir=root/fold_id; fold_dir.mkdir(exist_ok=False)
        norms[fold_id]=fold["normalization"]
        (fold_dir/"normalization.json").write_text(json.dumps(fold["normalization"],indent=2)+"\n")
        (fold_dir/"membership.json").write_text(json.dumps({"source_train_cards":fold["source_train_cards"],"source_validation_card":fold["source_validation_card"],
           "target_adaptation":"target training collections","evaluation":"designated evaluation collections"},indent=2)+"\n")
        source_dir=fold_dir/"source"
        train_cfg={"seed":SEED+i,"learning_rate":LEARNING_RATE,"batch_size":BATCH_SIZE,"max_epochs":MAX_EPOCHS_SOURCE,"patience":EARLY_STOPPING_PATIENCE}
        source_model,source_history=train_source_model(fold["x_train"],fold["y_train"],fold["x_val"],fold["y_val"],source_dir,train_cfg)

        adapt_dir=fold_dir/"adapted"
        adapt_cfg={"seed":SEED+i,"learning_rate":LEARNING_RATE,"lambda_coral":LAMBDA_CORAL,"batch_size":BATCH_SIZE,
                   "max_epochs":MAX_EPOCHS_ADAPT,"patience":EARLY_STOPPING_PATIENCE}
        adapted_model,adapt_history=adapt_joint_multi_target(source_dir/"source_model.keras",fold["x_train"],fold["y_train"],
                    fold["target_x"],fold["x_val"],fold["y_val"],adapt_dir,adapt_cfg)
        histories[fold_id]={"source":source_history,"adapted":adapt_history}
        for model_name,model in (("source",source_model),("adapted",adapted_model)):
            for d,spec in EVALUATION_PATHS.items():
                r=evaluate_model_on_collection(model,d,spec["cards"],fold["normalization"],fold_dir/"evaluation",model_name)
                records.append({"fold":fold_id,"model":model_name,"domain":d,"n":r["n"],"accuracy":r["accuracy"]})
        del source_model,adapted_model,fold
    metric_df=pd.DataFrame(records);metric_df.to_csv(root/"per_fold_metrics.csv",index=False)
    summary=metric_df.groupby(["model","domain"]).accuracy.agg(["mean","std","count"]).reset_index()
    summary["mean_percent"]=100*summary["mean"];summary["sample_sd_pp"]=100*summary["std"]
    summary.to_csv(root/"five_card_split_summary.csv",index=False)
    combined = []
    for (fold_id, model_name), group in metric_df.groupby(["fold", "model"]):
        combined.append({"fold": fold_id, "model": model_name,
                         "sample_weighted_accuracy": float(np.average(group.accuracy, weights=group.n)),
                         "macro_domain_accuracy": float(group.accuracy.mean())})
    pd.DataFrame(combined).to_csv(root/"combined_metrics.csv", index=False)
    (root/"histories.json").write_text(json.dumps(histories,indent=2)+"\n")
    manifest={"status":"completed exploratory evaluation","split_source":SPLIT_ORIGIN,"folds":FOLD_IDS,
      "mean_sd":"five source-card validation folds; sample SD ddof=1; same final evaluation collections reused each fold",
      "overlap_status":overlap_status,"known_overlap_audit":KNOWN_OVERLAP_AUDIT,"model_config":MODEL_CONFIG,
      "training_config":{"seed":SEED,"learning_rate":LEARNING_RATE,"lambda_coral":LAMBDA_CORAL,"batch_size":BATCH_SIZE,
      "max_epochs_source":MAX_EPOCHS_SOURCE,"max_epochs_adapt":MAX_EPOCHS_ADAPT},"normalization_by_fold":norms}
    (root/"run_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
    return root,metric_df,summary

FULL_RUN_RESULT=None
if RUN_FULL_EXPERIMENT:
    FULL_RUN_RESULT=run_five_card_split_experiment()
else:
    print("Full training disabled: RUN_FULL_EXPERIMENT=False.")


In [ ]:
def load_complete_model(path):
    model=tf.keras.models.load_model(path,compile=False)
    if model.output_shape[-1]!=12 or model.get_layer("shared_encoder").output_shape[-1]!=64:
        raise ValueError("Loaded model does not satisfy the complete 12-logit/64D encoder contract")
    return model


In [ ]:
def evaluate_saved_predictions(predictions_npz):
    with np.load(predictions_npz,allow_pickle=False) as z:
        y=z["true_labels"]; pred=z["predictions"]; prob=z["probabilities"]; ids=z["sample_ids"]
        assert len(ids)==len(y)==len(pred)==len(prob)
        assert prob.shape[1]==12 and np.allclose(prob.sum(1),1.0,atol=1e-6)
        return {"n":len(y),"accuracy":float(np.mean(y==pred)),"confusion":confusion_matrix(y,pred,labels=np.arange(12))}


In [ ]:
def summarize_five_card_splits(per_fold_metrics):
    df=pd.DataFrame(per_fold_metrics)
    out=df.groupby(["model","domain"]).accuracy.agg(mean="mean",sample_sd=lambda x:x.std(ddof=1),folds="count").reset_index()
    out["accuracy_mean_sd_percent"]=[f"{100*m:.2f} ± {100*s:.2f}%" for m,s in zip(out["mean"],out["sample_sd"])]
    return out


In [ ]:
def save_tsne_coordinates(embeddings,labels,sample_ids,domains,out_path,seed=SEED,max_rows=3000):
    n=len(embeddings)
    if n < 3 or max_rows < 3: raise ValueError("t-SNE requires at least three selected rows")
    rng=np.random.default_rng(seed)
    take=np.arange(n) if n<=max_rows else np.sort(rng.choice(n,size=max_rows,replace=False))
    z=np.asarray(embeddings)[take]
    perplexity=min(30.0,max(2.0,(len(z)-1)/3.0))
    coords=TSNE(n_components=2,random_state=seed,perplexity=perplexity,init="pca",learning_rate="auto").fit_transform(z)
    np.savez_compressed(out_path,coordinates=coords,labels=np.asarray(labels)[take],sample_ids=np.asarray(sample_ids)[take],
                        domains=np.asarray(domains)[take],source_indices=take,settings=np.asarray(json.dumps({"n_components":2,"seed":seed,"perplexity":perplexity,"init":"pca","learning_rate":"auto","max_rows":max_rows})))
    return Path(out_path)

def plot_saved_tsne(coords_path,output_path=None):
    with np.load(coords_path,allow_pickle=False) as z:
        xy=z["coordinates"]; labels=z["labels"]; domains=z["domains"]
    fig,ax=plt.subplots(figsize=(9,7))
    for d in np.unique(domains):
        ix=domains==d;ax.scatter(xy[ix,0],xy[ix,1],s=8,alpha=.65,label=str(d))
    ax.set_title("Saved encoder t-SNE coordinates");ax.legend(markerscale=2);fig.tight_layout()
    if output_path: fig.savefig(output_path,dpi=160)
    plt.show();return fig



In [ ]:
environment = {"python": platform.python_version(), "numpy": np.__version__,
               "pandas": pd.__version__, "tensorflow": tf.__version__, "scikit_learn": sklearn.__version__}
provenance = {"data_root": str(DATA_ROOT), "output_directory": str(PROJECT_DIR),
              "split_origin": SPLIT_ORIGIN, "source_validation_folds": FOLD_SOURCE_CARDS,
              "model_config": MODEL_CONFIG,
              "known_historical_overlap": KNOWN_OVERLAP_AUDIT,
              "domain_paths": {"adaptation": ADAPTATION_PATHS, "evaluation": EVALUATION_PATHS},
              "full_run_enabled": RUN_FULL_EXPERIMENT,
              "environment": environment}
with (PROJECT_DIR / "notebook_provenance.json").open("x") as f:
    json.dump(provenance, f, indent=2)
INVENTORY.to_csv(PROJECT_DIR / "dataset_inventory.csv", index=False)
CARD_SPLITS.to_csv(PROJECT_DIR / "source_card_splits.csv", index=False)
print("Output directory:", PROJECT_DIR)


In [ ]:
RUN_SAVED_INFERENCE = False
SAVED_MODEL_PATH = None
SAVED_NORMALIZATION_PATH = None
INFERENCE_BATCH_SIZE = 128
TSNE_MAX_ROWS = 3000
TSNE_RANDOM_STATE = 123
PLOT_SIZE = (2.10, 2.05)
SHOW_ZERO_VALUES = True
SAVED_PREDICTION_DIR = None
SAVED_TSNE_PATH = None


In [ ]:
def find_model_normalization(model_path, explicit_path=None):
    candidates = [Path(explicit_path)] if explicit_path is not None else [
        model_path.parent / "normalization.json",
        model_path.parent.parent / "normalization.json",
        model_path.parent.parent / "configs" / "preprocessing.json",
    ]
    for path in candidates:
        if path.is_file():
            stats = json.loads(path.read_text())
            if "global_mean" in stats and "global_std" in stats:
                stats = {**stats, "mean": stats["global_mean"], "std": stats["global_std"]}
            if "mean" not in stats or "std" not in stats:
                raise ValueError(f"Missing saved mean/std: {path}")
            order = stats.get("class_order", CLASS_ORDER)
            if order != CLASS_ORDER:
                raise ValueError("Saved normalization class order differs from B0-B11")
            return path, stats
    raise FileNotFoundError("Set SAVED_NORMALIZATION_PATH to this model's saved source-training normalization")

def infer_saved_dl(model_path, normalization_path=None):
    model_path = Path(model_path).expanduser()
    model = load_complete_model(model_path)
    norm_path, norm = find_model_normalization(model_path, normalization_path)
    root = OUTPUT_ROOT / ("inference_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8])
    root.mkdir(parents=True, exist_ok=False)
    records = []
    for domain, spec in EVALUATION_PATHS.items():
        raw = load_domain_magnitudes(domain, spec["cards"], evaluation=True)
        x = apply_source_normalizer(raw["magnitude"], norm)
        logits, probabilities, embeddings = predict_logits_probabilities_embeddings(model, x, batch_size=INFERENCE_BATCH_SIZE)
        predictions = logits.argmax(axis=1)
        if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6):
            raise ValueError("Invalid probability normalization")
        np.savez_compressed(root / f"DL_{domain}_predictions.npz",
                            sample_ids=raw["sample_id"], true_labels=raw["y"], predictions=predictions,
                            logits=logits, probabilities=probabilities, embeddings=embeddings,
                            class_order=np.asarray(CLASS_ORDER), card_folder=raw["card_folder"],
                            reader=raw["reader"], row_order=raw["row_order"])
        records.append({"domain": domain, "n": len(predictions),
                        "accuracy_percent": 100 * float(np.mean(predictions == raw["y"]))})
        del raw, x
    pd.DataFrame(records).to_csv(root / "accuracy.csv", index=False)
    (root / "inference_config.json").write_text(json.dumps({
        "checkpoint": str(model_path.resolve()), "normalization_path": str(norm_path.resolve()),
        "normalization": norm, "class_order": CLASS_ORDER,
        "data_root": str(DATA_ROOT), "domains": EVALUATION_PATHS,
        "batch_size": INFERENCE_BATCH_SIZE}, indent=2) + "\n")
    return root, pd.DataFrame(records)

INFERENCE_RESULT_DIR = None
if RUN_SAVED_INFERENCE:
    if SAVED_MODEL_PATH is None:
        raise ValueError("Set SAVED_MODEL_PATH to source_model.keras or adapted_model.keras")
    INFERENCE_RESULT_DIR, INFERENCE_METRICS = infer_saved_dl(SAVED_MODEL_PATH, SAVED_NORMALIZATION_PATH)
    display(INFERENCE_METRICS)
    print(INFERENCE_RESULT_DIR)


In [ ]:
from matplotlib.lines import Line2D

BUTTON_COLORS = np.array([
    [213,38,39], [254,126,13], [254,216,46], [43,161,94],
    [65,196,210], [46,90,210], [139,78,35], [231,61,153],
    [105,0,199], [101,118,24], [47,47,47], [19,124,138]], dtype=float) / 255.0
DOMAIN_MARKERS = {"Source": "o", "CC": "s", "CR": "^", "CCR": "D"}
FIGURE_STYLE = {"font.family": "serif", "font.serif": ["DejaVu Serif"],
                "pdf.fonttype": 42, "ps.fonttype": 42,
                "axes.labelsize": 8, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5}

def save_dl_figure(fig, output_stem):
    stem = Path(output_stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    for suffix in (".pdf", ".png"):
        path = stem.with_suffix(suffix)
        if path.exists():
            raise FileExistsError(path)
        fig.savefig(path, dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)

def plot_dl_confusion(true_labels, predictions, output_stem):
    y, pred = np.asarray(true_labels), np.asarray(predictions)
    if y.ndim != 1 or y.shape != pred.shape or len(y) == 0:
        raise ValueError("Expected non-empty paired label/prediction vectors")
    if not np.isin(y, range(12)).all() or not np.isin(pred, range(12)).all():
        raise ValueError("Labels must be B0-B11 class indices")
    counts = confusion_matrix(y, pred, labels=np.arange(12))
    support = counts.sum(axis=1, keepdims=True)
    values = np.divide(counts, support, out=np.zeros_like(counts, dtype=float), where=support != 0)
    stem = Path(output_stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(stem) + "_arrays.npz", counts=counts, row_normalized=values,
                        class_order=np.asarray(CLASS_ORDER), support=support.ravel())
    pd.DataFrame(values, index=CLASS_ORDER, columns=CLASS_ORDER).to_csv(str(stem) + "_values.csv")
    with plt.rc_context(FIGURE_STYLE):
        fig = plt.figure(figsize=PLOT_SIZE)
        ax = fig.add_axes([0.20, 0.225, 0.66, 0.676])
        cmap = plt.get_cmap("Blues")
        mesh = ax.imshow(values, vmin=0, vmax=1, cmap=cmap, aspect="auto", interpolation="none")
        ax.set_xticks(range(12), labels=CLASS_ORDER, rotation=90)
        ax.set_yticks(range(12), labels=CLASS_ORDER)
        ax.set_xlabel("Predicted button", labelpad=3)
        ax.set_ylabel("True button", labelpad=2)
        ax.tick_params(length=0, pad=1)
        for row in range(12):
            for col in range(12):
                value = values[row, col]
                if value == 0 and not SHOW_ZERO_VALUES:
                    continue
                rgb = np.asarray(cmap(value)[:3])
                color = "white" if rgb @ np.array([0.2126,0.7152,0.0722]) < 0.44 else "black"
                ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                        fontsize=3.6, fontfamily="DejaVu Sans", color=color)
        cax = fig.add_axes([0.878, 0.225, 0.022, 0.676])
        cb = fig.colorbar(mesh, cax=cax, ticks=[0,0.5,1])
        cb.ax.tick_params(labelsize=6.5, length=1.5, width=0.5, pad=1.5)
        save_dl_figure(fig, stem)
    return counts, values

def plot_dl_tsne(coordinates_path, output_stem):
    with np.load(coordinates_path, allow_pickle=False) as z:
        xy, labels, domain_labels = z["coordinates"], z["labels"], z["domains"]
    if xy.shape != (len(labels), 2) or len(domain_labels) != len(labels):
        raise ValueError("Saved coordinate/label/domain alignment mismatch")
    with plt.rc_context(FIGURE_STYLE):
        fig = plt.figure(figsize=PLOT_SIZE)
        ax = fig.add_axes([0.20, 0.37, 0.74, 0.57])
        domain_order = [d for d in DOMAIN_MARKERS if d in set(domain_labels)]
        domain_order += [d for d in np.unique(domain_labels) if d not in domain_order]
        for domain in domain_order:
            ix = domain_labels == domain
            ax.scatter(xy[ix,0], xy[ix,1], c=BUTTON_COLORS[labels[ix].astype(int)],
                       marker=DOMAIN_MARKERS.get(str(domain), "o"), s=2.5,
                       alpha=0.7, linewidths=0, rasterized=True)
        ax.set_xlabel("t-SNE 1", labelpad=2)
        ax.set_ylabel("t-SNE 2", labelpad=2)
        ax.tick_params(direction="in", length=2, width=0.5, pad=1.5)
        buttons = [Line2D([],[],marker="o",linestyle="none",markerfacecolor=BUTTON_COLORS[b],
                          markeredgecolor="none",markersize=2.4,label=CLASS_ORDER[b]) for b in range(12)]
        domains = [Line2D([],[],marker=DOMAIN_MARKERS.get(str(d),"o"),linestyle="none",
                          color="black",markersize=2.4,label=str(d)) for d in domain_order]
        fig.legend(handles=domains, loc="lower center", bbox_to_anchor=(0.56,0.13),
                   ncol=len(domains), fontsize=5.8, frameon=False, handletextpad=0.2, columnspacing=0.6)
        fig.legend(handles=buttons, loc="lower center", bbox_to_anchor=(0.51,0.015),
                   ncol=6, fontsize=5.2, frameon=False, handletextpad=0.1, columnspacing=0.25)
        save_dl_figure(fig, output_stem)

def plot_dl_saved_predictions(prediction_files, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    all_features, all_labels, all_ids, all_domains = [], [], [], []
    summary = []
    for domain, filename in prediction_files.items():
        with np.load(filename, allow_pickle=False) as z:
            y, pred = z["true_labels"], z["predictions"]
            embedding, sample_ids = z["embeddings"], z["sample_ids"]
            if not np.array_equal(z["class_order"], CLASS_ORDER):
                raise ValueError("Class order mismatch")
            if embedding.shape != (len(y),64) or len(sample_ids) != len(y):
                raise ValueError("Feature/sample alignment mismatch")
            counts, _ = plot_dl_confusion(y, pred, output_dir / f"DL_{domain}_Confusion_Matrix")
            summary.append({"domain":domain, "n":len(y), "accuracy_percent":100*float(np.mean(y==pred))})
            all_features.append(embedding); all_labels.append(y); all_ids.append(sample_ids)
            all_domains.append(np.repeat(domain,len(y)))
    coordinates_path = output_dir / "DL_tSNE_coordinates.npz"
    save_tsne_coordinates(np.concatenate(all_features), np.concatenate(all_labels),
                          np.concatenate(all_ids), np.concatenate(all_domains), coordinates_path,
                          seed=TSNE_RANDOM_STATE, max_rows=TSNE_MAX_ROWS)
    plot_dl_tsne(coordinates_path, output_dir / "DL_tSNE")
    pd.DataFrame(summary).to_csv(output_dir / "accuracy.csv", index=False)
    return output_dir


In [ ]:
PLOT_RESULT_DIRS = []
if INFERENCE_RESULT_DIR is not None:
    files = {d: INFERENCE_RESULT_DIR / f"DL_{d}_predictions.npz" for d in ("CC","CR","CCR")}
    plot_dir = INFERENCE_RESULT_DIR / ("figures_" + uuid.uuid4().hex[:8])
    PLOT_RESULT_DIRS.append(plot_dl_saved_predictions(files, plot_dir))
if FULL_RUN_RESULT is not None:
    full_root, _, _ = FULL_RUN_RESULT
    for fold_id in FOLD_IDS:
        for model_name in ("source", "adapted"):
            files = {d: full_root / fold_id / "evaluation" / f"{model_name}_{d}_predictions.npz"
                     for d in ("CC","CR","CCR")}
            plot_dir = full_root / fold_id / (model_name + "_figures_" + uuid.uuid4().hex[:8])
            PLOT_RESULT_DIRS.append(plot_dl_saved_predictions(files, plot_dir))
print("Figure directories:", PLOT_RESULT_DIRS)


In [ ]:
if SAVED_PREDICTION_DIR is not None:
    cached = Path(SAVED_PREDICTION_DIR)
    files = {d: cached / f"DL_{d}_predictions.npz" for d in ("CC","CR","CCR")}
    destination = cached / ("confusion_plots_" + uuid.uuid4().hex[:8])
    destination.mkdir(parents=True, exist_ok=False)
    for domain, filename in files.items():
        with np.load(filename, allow_pickle=False) as z:
            plot_dl_confusion(z["true_labels"], z["predictions"], destination / f"DL_{domain}_Confusion_Matrix")
    print(destination)
if SAVED_TSNE_PATH is not None:
    path = Path(SAVED_TSNE_PATH)
    plot_dl_tsne(path, path.parent / ("DL_tSNE_replot_" + uuid.uuid4().hex[:8]))
